# 🔎 34 · Explainability & Trustworthy AI for Bioimage Models
### Why "it worked on my test set" is not enough to trust a model on new data

**Part J · Research frontier** · Module 34 of 37 (frontier) · builds on 16, 19

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/34_explainability_and_trustworthy_ai.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (every demo finishes in well under two minutes) |
| **Core frameworks** | NumPy, Matplotlib, PyTorch |
| **Prerequisites** | Module 16 (deep learning classification), Module 19 (deep segmentation) |
| **Builds on** | Modules 16, 19 |
| **Research frontier covered** | Saliency/attribution, failure-mode auditing, dataset and domain shift, reproducibility pitfalls |
| **Estimated time** | 80–100 min |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** Because every label, every "lab" condition and every duplicate image is produced by code with known ground truth, each failure mode demonstrated below is quantified exactly, not estimated.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why a high test-set accuracy is necessary but not sufficient evidence that a model can
  be trusted on new data.
- Compute two attribution methods — gradient-based saliency and occlusion sensitivity — for a CNN
  classifier, and sanity-check that an attribution map actually highlights the object of interest
  rather than the background.
- Build and measure an out-of-distribution (OOD) test set, and quantify how much accuracy drops
  when a model meets a different "lab" (imaging condition) than the one it trained on.
- Reproduce, in miniature, the single most common way bioimage-analysis evaluations are
  accidentally inflated: train/test leakage from near-duplicate images, and threshold-tuning on
  the test set itself.
- List concrete practices that defend against each of these failure modes in your own work.

## 🗺️ Contents

1. Why a test-set number is not the whole story
2. Saliency and attribution: what is the model actually looking at?
3. Failure-mode auditing: building an out-of-distribution test set
4. Dataset and domain shift between labs, scopes and settings
5. Reproducibility pitfalls: leakage and threshold-tuning
6. 🧭 From toy to real data

## 🧫 The biological question

Suppose a collaborator hands you a trained classifier with "97% test accuracy" and asks whether
they can use it to screen a new batch of slides from a different microscope in a different room.
The honest answer requires more than the number 97%: it requires knowing *what* that 97% was
measured on, *how* the test set was built, and *what* the model is actually responding to in each
image. This module is about the tools that turn "I trust this model" from a feeling into something
you can check — and about the specific, well-documented ways bioimage-analysis evaluations go
wrong without anyone noticing.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 26
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

The exact same helper functions used throughout this course, copied verbatim so this notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · Why a test-set number is not the whole story

A test-set accuracy answers one narrow question: *how well did this model do on images drawn from
the same distribution as its training data?* It says nothing about what happens on images that
are not from that distribution — a different microscope, a different cell line, a batch with more
debris — and it says nothing about *why* the model got any particular image right, which matters
because a model can get the right answer for the wrong reason (and then get the next, slightly
different image wrong for the same reason). The rest of this notebook builds concrete, checkable
evidence for both of these gaps.

---

## 2 · Saliency and attribution: what is the model actually looking at?

We start, as in Module 32, with a small CNN classifying single-cell shape patches. This time the
question is not "how confident is it?" but "**which pixels** drove that prediction?" Two classic,
easy-to-implement attribution methods:

- **Gradient-based saliency**: the gradient of the predicted class score with respect to each
  input pixel. A large gradient magnitude at a pixel means a small change there would have moved
  the prediction a lot.
- **Occlusion sensitivity**: slide a small gray patch over the image; wherever covering the image
  causes the predicted score to drop the most, that region mattered the most. More expensive, but
  conceptually simpler and does not require backpropagating through the model.

In [ ]:
# ── A small classifier, trained exactly as in Module 32 ──
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
SHAPE_TO_IDX = {s: i for i, s in enumerate(SHAPE_CLASSES)}
PATCH_SIZE = 48    # the classifier's input size
GEN_SIZE = 160     # the simulator canvas size (shape detail needs a reasonably large canvas)


def crop_box(sim, margin=1.7):
    """The (y0, y1, x0, x1) tight crop box around a simulated single-cell image's known bounding
    box — shared by the image crop below and, in §2, the matching ground-truth mask crop."""
    img = sim["image"]
    ymin, xmin, ymax, xmax = sim["boxes"][0]
    cy, cx = (ymin + ymax) / 2, (xmin + xmax) / 2
    half = max(6.0, max(ymax - ymin, xmax - xmin) / 2 * margin)
    y0, y1 = int(max(0, cy - half)), int(min(img.shape[0], cy + half))
    x0, x1 = int(max(0, cx - half)), int(min(img.shape[1], cx + half))
    return y0, y1, x0, x1


def crop_and_resize_cell(sim, patch_size=PATCH_SIZE, margin=1.7):
    """Crop a simulated single-cell image tightly around its (known) bounding box and resize to
    a fixed patch_size — the same "detect, then crop-and-classify" step a real pipeline uses."""
    y0, y1, x0, x1 = crop_box(sim, margin)
    crop = sim["image"][y0:y1, x0:x1]
    zoom_factors = (patch_size / crop.shape[0], patch_size / crop.shape[1])
    return ndi.zoom(crop, zoom_factors, order=1)


def make_shape_dataset(n_per_class, seed, shape_classes=SHAPE_CLASSES, noise_sigma=0.05, psf_sigma=1.2):
    """Returns (images, labels, seeds_used) — the per-sample seed is kept so any single example
    can be regenerated exactly later (e.g. to recompute its ground-truth mask, in §2)."""
    images, labels, seeds_used = [], [], []
    s = seed * 100_003
    for shape in shape_classes:
        for _ in range(n_per_class):
            sim = simulate_nuclei_image(n_cells=1, size=GEN_SIZE, shape_classes=(shape,),
                                         noise_sigma=noise_sigma, psf_sigma=psf_sigma, seed=s)
            images.append(crop_and_resize_cell(sim)); labels.append(SHAPE_TO_IDX[shape])
            seeds_used.append(s); s += 1
    X = np.stack(images)[:, None, :, :].astype(np.float32)
    y = np.array(labels, dtype=np.int64)
    seeds_used = np.array(seeds_used)
    perm = rng.permutation(len(y))
    return X[perm], y[perm], seeds_used[perm]


class TinyShapeNet(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc1 = nn.Linear(16 * 12 * 12, 32)
        self.fc2 = nn.Linear(32, n_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.relu(self.fc1(x.flatten(1)))
        return self.fc2(x)


def train_classifier(X, y, n_epochs=180, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    model = TinyShapeNet()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    for _ in range(n_epochs):
        opt.zero_grad()
        loss = F.cross_entropy(model(Xt), yt)
        loss.backward()
        opt.step()
    model.eval()
    return model


X_train, y_train, _ = make_shape_dataset(50, seed=1)
X_test, y_test, seeds_test = make_shape_dataset(20, seed=999)
model_a = train_classifier(X_train, y_train, seed=SEED)
with torch.no_grad():
    acc = (model_a(torch.from_numpy(X_test)).argmax(1).numpy() == y_test).mean()
print(f"test accuracy on the training distribution ('Lab A'): {acc:.1%}")

In [ ]:
# ── Gradient-based saliency and occlusion attribution ──
def gradient_saliency(model, image, target_class):
    x = torch.from_numpy(image[None, None]).float().requires_grad_(True)
    score = model(x)[0, target_class]
    score.backward()
    return x.grad[0, 0].detach().numpy()


def occlusion_attribution(model, image, target_class, patch=6, stride=3):
    with torch.no_grad():
        base = torch.from_numpy(image[None, None]).float()
        base_score = F.softmax(model(base), dim=1)[0, target_class].item()
    baseline_val = float(image.mean())
    H, W = image.shape
    attribution = np.zeros((H, W)); counts = np.zeros((H, W))
    for y0 in range(0, H - patch + 1, stride):
        for x0 in range(0, W - patch + 1, stride):
            occ = image.copy()
            occ[y0:y0 + patch, x0:x0 + patch] = baseline_val
            with torch.no_grad():
                score = F.softmax(model(torch.from_numpy(occ[None, None]).float()), dim=1)[0, target_class].item()
            attribution[y0:y0 + patch, x0:x0 + patch] += (base_score - score)
            counts[y0:y0 + patch, x0:x0 + patch] += 1
    return attribution / np.maximum(counts, 1e-9)


# Pick one clean, correctly classified test example to inspect
example_idx = np.where(model_a(torch.from_numpy(X_test)).argmax(1).numpy() == y_test)[0][0]
example_img = X_test[example_idx, 0]
example_label = y_test[example_idx]
# `make_shape_dataset` kept the exact seed used for every image (seeds_test); regenerate that
# SAME simulated cell so we can crop-and-resize its exact ground-truth mask, the same way as the image.
sim_example = simulate_nuclei_image(n_cells=1, size=GEN_SIZE,
                                     shape_classes=(SHAPE_CLASSES[example_label],),
                                     noise_sigma=0.05, seed=int(seeds_test[example_idx]))
y0, y1, x0, x1 = crop_box(sim_example)
mask_crop = (sim_example["instance_mask"][y0:y1, x0:x1] > 0).astype(float)
zf = (PATCH_SIZE / mask_crop.shape[0], PATCH_SIZE / mask_crop.shape[1])
example_mask = ndi.zoom(mask_crop, zf, order=0) > 0.5

saliency = np.abs(gradient_saliency(model_a, example_img, example_label))
occlusion = np.clip(occlusion_attribution(model_a, example_img, example_label), 0, None)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
show_image(axes[0], example_img, f"input ({SHAPE_CLASSES[example_label]})")
show_image(axes[1], saliency, "gradient saliency", cmap="inferno")
show_image(axes[2], occlusion, "occlusion attribution", cmap="inferno")
fig.tight_layout()
plt.show()

In [ ]:
# ── 🧪 Ground-truth check, averaged over many examples (not just one image) ──
def mass_fraction_in_mask(attribution, mask):
    pos = np.clip(attribution, 0, None)
    return pos[mask].sum() / (pos.sum() + 1e-9)


def ground_truth_mask_for(idx):
    """Regenerate test example `idx`'s exact simulated cell (same seed) and crop/resize its
    instance mask exactly the way crop_and_resize_cell treats the image, so the two line up."""
    label = y_test[idx]
    sim = simulate_nuclei_image(n_cells=1, size=GEN_SIZE, shape_classes=(SHAPE_CLASSES[label],),
                                 noise_sigma=0.05, seed=int(seeds_test[idx]))
    y0, y1, x0, x1 = crop_box(sim)
    mask_crop = (sim["instance_mask"][y0:y1, x0:x1] > 0).astype(float)
    zf = (PATCH_SIZE / mask_crop.shape[0], PATCH_SIZE / mask_crop.shape[1])
    return ndi.zoom(mask_crop, zf, order=0) > 0.5


correct_idx = np.where(model_a(torch.from_numpy(X_test)).argmax(1).numpy() == y_test)[0][:15]
area_fracs, sal_fracs, occ_fracs = [], [], []
for idx in correct_idx:
    mask = ground_truth_mask_for(idx)
    img = X_test[idx, 0]
    sal = np.abs(gradient_saliency(model_a, img, y_test[idx]))
    occ = occlusion_attribution(model_a, img, y_test[idx])
    area_fracs.append(mask.mean())
    sal_fracs.append(mass_fraction_in_mask(sal, mask))
    occ_fracs.append(mass_fraction_in_mask(occ, mask))

print(f"averaged over {len(correct_idx)} correctly classified test images:")
print(f"  the cell occupies                                   {np.mean(area_fracs):.1%} of the image")
print(f"  gradient saliency mass inside the true cell mask     {np.mean(sal_fracs):.1%}")
print(f"  occlusion attribution mass inside the true cell mask {np.mean(occ_fracs):.1%}")

🧪 **Ground-truth check.** Because we built every image ourselves, we know exactly
which pixels are "cell" and which are "background" for each one. If an attribution method is doing
something sensible, the fraction of its (positive) attribution mass that falls inside the true
cell mask should sit noticeably **above** the cell's raw area fraction — the model should be
looking at the object, not spreading its attention uniformly (or worse, toward the background).
Averaged over several examples, occlusion attribution should come out clearly above the area
fraction here, while raw input-gradient saliency typically lands much closer to it — a concrete,
checkable illustration of why gradient saliency has a reputation for being noisy, and why better
methods (Grad-CAM, §6) were developed on top of it. If *neither* method beats the area fraction on
your own re-run, that is itself a useful (if less comfortable) finding worth reporting honestly
rather than discarding.

✏️ **Your turn.** Try a larger `patch`/`stride` for the occlusion sweep (coarser,
faster) and a smaller one (finer, slower) — how much does the mass fraction change? Also try
computing `gradient_saliency` from a *randomly initialized, untrained* copy of `TinyShapeNet` —
you should see its mass fraction collapse toward the raw area fraction, since an untrained network
has no learned reason to look at the cell at all. That comparison is itself a sanity check: an
attribution method should behave differently for a trained vs. an untrained model.

---

## 3 · Failure-mode auditing: building an out-of-distribution test set

The 97%-on-my-test-set number in §2 was measured on images drawn from exactly the distribution
the model trained on. Let's build a second test set that differs in one controlled way — a
blurrier, noisier "microscope" — and measure the same model's accuracy on it.

In [ ]:
# ── 🧪 A controlled out-of-distribution test set ──
X_ood, y_ood, _ = make_shape_dataset(20, seed=555, noise_sigma=0.14, psf_sigma=2.3)

with torch.no_grad():
    acc_ood = (model_a(torch.from_numpy(X_ood)).argmax(1).numpy() == y_ood).mean()
print(f"test accuracy, training distribution ('Lab A')        : {acc:.1%}")
print(f"test accuracy, out-of-distribution ('blurrier scope')  : {acc_ood:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(6, 3.2))
show_image(axes[0], X_test[0, 0], "Lab A condition")
show_image(axes[1], X_ood[0, 0], "out-of-distribution condition")
fig.tight_layout()
plt.show()

🔍 **What just happened?** Nothing about the *task* changed — the same six shape
classes, generated by the same simulator, with exactly known labels — only the imaging condition
changed (more blur, more noise). A sizeable accuracy drop here, on a controlled synthetic problem
where we can rule out every other explanation, is the cleanest possible demonstration that a
model's test accuracy is a property of *the test set it was measured on*, not a property of the
model in the abstract.

---

## 4 · Dataset and domain shift between labs, scopes and settings

What we built in §3 — a model that trained on one imaging condition and is evaluated on another —
is, in practice, the single most common way bioimage-analysis models fail silently. It almost
never announces itself as "domain shift"; it shows up as "the model works great in our hands but
falls apart when our collaborator's lab tries it," because their microscope, their staining
protocol, or their camera settings differ from yours in ways neither of you thought to control
for. The fix that generalizes best is simply **more data, from more conditions**; a cheaper partial
fix, when you have at least a little labeled data from the new condition, is to continue training
(fine-tune) on it.

In [ ]:
# ── A small amount of labeled "Lab B" data goes a long way ──
X_finetune, y_finetune, _ = make_shape_dataset(8, seed=321, noise_sigma=0.14, psf_sigma=2.3)  # 8/class


def finetune(model, X, y, n_epochs=15, lr=5e-4, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    model.train()
    for _ in range(n_epochs):
        opt.zero_grad()
        loss = F.cross_entropy(model(Xt), yt)
        loss.backward()
        opt.step()
    model.eval()
    return model


import copy
model_finetuned = finetune(copy.deepcopy(model_a), X_finetune, y_finetune, seed=SEED)
with torch.no_grad():
    acc_ood_finetuned = (model_finetuned(torch.from_numpy(X_ood)).argmax(1).numpy() == y_ood).mean()

print(f"Lab B accuracy, model trained only on Lab A              : {acc_ood:.1%}")
print(f"Lab B accuracy, after fine-tuning on {len(y_finetune)} labeled Lab B examples: {acc_ood_finetuned:.1%}")

🧪 **Ground-truth check.** The fine-tuned model saw only `len(y_finetune)` labeled
examples from the new condition — a tiny fraction of the original training set — yet should
recover a meaningful chunk of the accuracy gap, because `X_ood`/`y_ood` are exactly labeled and
untouched by the fine-tuning data. This is the practical lesson: domain shift is not usually fixed
by a cleverer architecture, it is fixed by *data from the condition you will actually deploy on*,
and you often need far less of it than the original training set if you start from a model that
already works elsewhere.

---

## 5 · Reproducibility pitfalls: leakage and threshold-tuning

Two of the most common ways a bioimage-analysis evaluation ends up **optimistic** — reporting a
number that will not hold up on genuinely new data — have nothing to do with domain shift at all;
they are mistakes in how the evaluation itself was built.

**Train/test leakage** happens when information from a test example has, in some indirect way,
already been seen during training — most often because near-duplicate images (two crops of the
same cell, two frames of the same field of view, two technical replicates of the same sample) end
up on both sides of a random split. The model can then partly succeed by half-recognizing a
specific example rather than generalizing.

In [ ]:
# ── 🧪 Reproduce leakage directly: duplicate "objects", then split two different ways ──
def make_object_with_duplicates(seed, shape_classes=SHAPE_CLASSES, n_per_class=30,
                                 duplicate_noise_sigma=0.01):
    """SIMULATED dataset where EVERY underlying object is imaged TWICE (e.g. a re-scan, or two
    adjacent z-planes of the same cell) with only a tiny extra noise difference between the two
    copies. Returns images, labels and an `object_id` per image so duplicates can be grouped."""
    images, labels, object_ids = [], [], []
    s = seed * 100_003
    oid = 0
    for shape in shape_classes:
        for _ in range(n_per_class):
            sim = simulate_nuclei_image(n_cells=1, size=GEN_SIZE, shape_classes=(shape,),
                                         noise_sigma=0.05, seed=s)
            base_image = crop_and_resize_cell(sim)
            for _copy in range(2):                       # two near-duplicate "images" per object
                dup = add_gaussian_noise(base_image, duplicate_noise_sigma,
                                          np.random.default_rng(s * 7 + _copy))
                images.append(dup); labels.append(SHAPE_TO_IDX[shape]); object_ids.append(oid)
            oid += 1
            s += 1
    return (np.stack(images)[:, None, :, :].astype(np.float32),
            np.array(labels, dtype=np.int64), np.array(object_ids))


X_dup, y_dup, obj_dup = make_object_with_duplicates(seed=42, n_per_class=18)
n_objects = obj_dup.max() + 1
print(f"{len(y_dup)} images total, from {n_objects} underlying objects (2 near-duplicates each)")

In [ ]:
# ── Split A: "leaky" — split INDIVIDUAL IMAGES at random, ignoring object_id ──
rng_split = np.random.default_rng(SEED)
idx = rng_split.permutation(len(y_dup))
n_test = len(y_dup) // 4
leaky_test_idx, leaky_train_idx = idx[:n_test], idx[n_test:]

model_leaky = train_classifier(X_dup[leaky_train_idx], y_dup[leaky_train_idx], seed=SEED)
with torch.no_grad():
    acc_leaky = (model_leaky(torch.from_numpy(X_dup[leaky_test_idx])).argmax(1).numpy()
                 == y_dup[leaky_test_idx]).mean()

# Split B: "grouped" — split by OBJECT, so both copies of an object stay on the same side
object_order = rng_split.permutation(n_objects)
n_test_objects = n_objects // 4
test_objects = set(object_order[:n_test_objects])
grouped_test_mask = np.array([o in test_objects for o in obj_dup])
model_grouped = train_classifier(X_dup[~grouped_test_mask], y_dup[~grouped_test_mask], seed=SEED)
with torch.no_grad():
    acc_grouped = (model_grouped(torch.from_numpy(X_dup[grouped_test_mask])).argmax(1).numpy()
                   == y_dup[grouped_test_mask]).mean()

print(f"'leaky' random split (duplicates can cross train/test): {acc_leaky:.1%} test accuracy")
print(f"'grouped' split (all of an object's images stay together): {acc_grouped:.1%} test accuracy")

🧪 **Ground-truth check.** Both splits used the same number of training and test
images, generated by the same simulator — the *only* difference is whether a near-duplicate of a
training image was allowed to land in the test set. If the leaky split reports a higher accuracy
than the grouped split, that gap **is** the size of the optimistic bias leakage introduces — and
it is a bias you only find by checking how the split was built, never by looking at the accuracy
number alone.

The second pitfall is **tuning a decision threshold on the test set itself.** It is
tempting, once you have a model's output scores, to pick whatever cutoff maximizes accuracy (or
F1, or whatever metric you report) on the data you have — but if that data is also what you report
performance on, you have let the test set leak into your choice of threshold, the same way a
duplicate image leaks into training.

This risk is largest exactly when a model's decisions are genuinely borderline — a confident
classifier's scores cluster near 0 or 1 and almost any reasonable threshold agrees, leaving little
room for threshold-shopping to matter. So to make the danger concrete, this demo deliberately uses
a less-polished classifier (fewer training epochs) on a harder imaging condition — precisely the
situation where a few extra points of accuracy from "optimizing" a threshold is most tempting to
report, and most misleading.

In [ ]:
# ── 🧪 Threshold-tuning on the test set vs. on a separate validation set ──
# A deliberately less-polished classifier (fewer epochs) on a harder imaging condition, so that
# threshold choice genuinely matters (see the markdown above for why).
X_train_hard, y_train_hard, _ = make_shape_dataset(50, seed=77, noise_sigma=0.12, psf_sigma=1.8)
model_borderline = train_classifier(X_train_hard, y_train_hard, n_epochs=30, seed=SEED)

X_val2, y_val2, _ = make_shape_dataset(15, seed=2024, noise_sigma=0.12, psf_sigma=1.8)   # VALIDATION
X_test2, y_test2, _ = make_shape_dataset(15, seed=3030, noise_sigma=0.12, psf_sigma=1.8)  # TEST, touched once

with torch.no_grad():
    probs_val = F.softmax(model_borderline(torch.from_numpy(X_val2)), dim=1).numpy()
    probs_test = F.softmax(model_borderline(torch.from_numpy(X_test2)), dim=1).numpy()

# Binary task for this demo: "is this a round nucleus?" accept if P(round) exceeds a threshold.
is_round_val = (y_val2 == SHAPE_TO_IDX["round"]).astype(int)
is_round_test = (y_test2 == SHAPE_TO_IDX["round"]).astype(int)
score_val, score_test = probs_val[:, SHAPE_TO_IDX["round"]], probs_test[:, SHAPE_TO_IDX["round"]]

thresholds = np.linspace(0.05, 0.95, 37)


def best_threshold(scores, labels):
    accs = [( (scores > t).astype(int) == labels ).mean() for t in thresholds]
    return thresholds[int(np.argmax(accs))], max(accs)


# (A) "p-hacked": tune directly on the test set, report the test set's own best accuracy
t_on_test, acc_reported_on_test = best_threshold(score_test, is_round_test)

# (B) honest: tune on the SEPARATE validation set, then evaluate that fixed threshold on test
t_on_val, _ = best_threshold(score_val, is_round_val)
acc_honest_on_test = ((score_test > t_on_val).astype(int) == is_round_test).mean()

print(f"threshold chosen BY LOOKING AT the test set   : {t_on_test:.2f}  "
      f"-> reports {acc_reported_on_test:.1%} on that SAME test set")
print(f"threshold chosen on a separate validation set : {t_on_val:.2f}  "
      f"-> scores {acc_honest_on_test:.1%} on the (never-looked-at-for-tuning) test set")

🔍 **What just happened?** Tuning directly on the test set finds whatever threshold
happens to work best for *that specific sample*, including quirks that will not repeat on the next
batch of images — this is the image-analysis version of p-hacking. The gap above is the size of
that optimistic bias: it exists purely because the same data was used to both choose the threshold
and report the accuracy. Choosing the threshold on a separate validation set and only then
touching the test set once gives an honest estimate of how that threshold will perform on new
data. In a real study, the test set should be looked at exactly once, after every modeling decision
(architecture, threshold, hyperparameters) has already been locked in on training and validation
data alone.

---

## 6 · 🧭 From toy to real data

### 📄 Research connection — Gradient-based saliency
Simonyan, K., Vedaldi, A. & Zisserman, A. (2014). *Deep Inside Convolutional Networks:
Visualising Image Classification Models and Saliency Maps.* ICLR Workshop. The original input-
gradient saliency method implemented by hand in §2.

### 📄 Research connection — Grad-CAM
Selvaraju, R. R. et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-
based Localization.* ICCV. A more robust, class-discriminative attribution method widely used in
practice, combining gradients with a convolutional layer's spatial feature maps rather than raw
input gradients.

### 📄 Research connection — a real, well-documented shortcut-learning failure
Zech, J. R. et al. (2018). *Variable generalization performance of a deep learning model to detect
pneumonia in chest radiographs: a cross-sectional study.* PLOS Medicine, 15(11). A widely cited,
real example of exactly §4's failure mode: models partly learned to recognize which *hospital*
(and even which specific X-ray machine) took an image — a pure batch-effect confound — rather than
the disease itself, and generalized far worse than their single-hospital test accuracy suggested.

### 📄 Research connection — reproducibility recommendations for bioimage analysis
Walsh, I. et al. (2021). *DOME: recommendations for supervised machine learning validation in
biology.* Nature Methods, 18. Concrete, field-specific recommendations — including exactly the
split-by-source-object and single-use-test-set practices demonstrated in §5 — for reporting
bioimage-analysis models in a way that is actually reproducible.

### 📄 Research connection — benchmarking pitfalls in medical/bioimage ML
Varoquaux, G. & Cheplygina, V. (2022). *Machine learning for medical imaging: methodological
failures and recommendations for the future.* npj Digital Medicine, 5. A broader survey of the
same family of evaluation mistakes (leakage, small/non-representative test sets, threshold-
tuning) across medical and biological imaging studies.

At production scale, every check in this notebook has a direct real-world counterpart: Grad-CAM
or integrated-gradients attribution maps audited by a domain expert before a model is trusted;
an explicit "external validation set" from a genuinely different site, scanner or batch, held out
from all model-development decisions; and a pre-registered evaluation protocol that fixes the test
set, the metric and the threshold *before* looking at test-set performance even once.

## 📌 Key takeaways

- A test-set accuracy only describes performance on data drawn from the training distribution —
  it says nothing, by itself, about new conditions.
- Attribution methods (gradient saliency, occlusion sensitivity) can be sanity-checked against
  known ground truth: a trustworthy attribution should land disproportionately on the object of
  interest, not the background.
- A deliberately constructed out-of-distribution test set quantifies exactly how much accuracy a
  model loses under a realistic imaging-condition shift — and a small amount of labeled data from
  the new condition can recover much of that gap via fine-tuning.
- Train/test leakage from near-duplicate images, and threshold-tuning on the test set itself, both
  inflate a reported accuracy in ways that only show up when you check *how the evaluation was
  built*, never from the accuracy number alone.
- The defenses are concrete and checkable: split by source object/subject, hold out a genuinely
  separate validation set for every tuning decision, and touch the test set exactly once.

## ✅ Checkpoint

**1. An occlusion attribution map for a correctly classified cell image puts most of its mass on a
corner of the image far from the cell. What are two different explanations for this, and how
would you distinguish them?**

<details><summary>Answer</summary>

One explanation is a real shortcut: the model has learned to key off something in the background
(e.g. an illumination gradient or a simulation artifact) that happens to correlate with the label
in this dataset, rather than the cell's shape itself — a genuine, if subtle, version of the
Zech et al. shortcut-learning failure. The other is a weakness of the attribution method itself
(occlusion with a fixed patch size can produce noisy or misleading maps, especially with a small
patch relative to object size). You would distinguish them by checking on many examples (not just
one), by trying a different attribution method (gradient saliency) and seeing if it agrees, and —
since this is a simulated dataset — by directly inspecting whether the background actually does
carry label-correlated information (for instance, whether a model trained on background-only crops
beats chance).
</details>

**2. Why does fine-tuning on a handful of Lab B examples (§4) typically recover accuracy faster
than training a brand-new model on the same small Lab B set from scratch?**

<details><summary>Answer</summary>

The fine-tuned model starts from weights that already encode a great deal of useful, transferable
structure (edges, blob/shape detectors) learned from the much larger Lab A training set — it only
needs to adjust for the *difference* between Lab A and Lab B, not relearn shape recognition from
nothing. A model trained from scratch on only a handful of Lab B examples has far too little data
to learn that structure on its own and would badly overfit.
</details>

**3. In the leakage experiment, which split's reported accuracy would you trust if a collaborator
handed you this model and this number with no further information — and what one question would
you ask before trusting either number?**

<details><summary>Answer</summary>

Neither number should be trusted blindly — the right question is "how was the test set built,
specifically: could any test example be a near-duplicate, replicate or crop of something in the
training set?" If the answer is "images were split by object/subject/field of view, with every
duplicate kept on one side," the grouped-split-style number is trustworthy; if the split was a
plain random shuffle of individual images, the higher (leaky-style) number should be treated as an
overestimate until re-measured with a proper grouped split.
</details>

## 🧠 Questions & discussion

- If attribution maps are themselves imperfect (as the Checkpoint question above shows), how much
  should you trust them as evidence a model is *not* shortcut-learning? What additional evidence,
  beyond a single attribution map on a single image, would make you more confident either way?
- The fine-tuning fix in §4 assumes you can get at least a little labeled data from the new
  condition. What would you do if you could not — if the new "Lab B" was a clinical site that
  would not share any labeled images at all, only unlabeled ones?
- Pre-registering an evaluation protocol (fixing the test set, metric and threshold before looking
  at test performance) is standard practice in clinical trials but still uncommon in most bioimage-
  analysis papers. What would make it easier or harder to adopt in a typical academic lab's
  workflow?

## 🚀 Bonus — going further

- Implement Grad-CAM (combine the gradient of the target class score with respect to the last
  convolutional layer's activations, rather than the raw input pixels) and compare its attribution
  maps and ground-truth mass-fraction scores (§2) to the two methods implemented here.
- Extend the leakage experiment to measure the gap as a function of how similar the duplicates are
  (vary `duplicate_noise_sigma` from near-zero to large) — at what point does the leaky split's
  advantage disappear?
- Design and run a third domain-shift scenario (e.g. a systematic brightness/contrast shift
  instead of blur+noise) and measure whether fine-tuning recovers accuracy equally well for every
  kind of shift, or whether some shifts are harder to adapt to than others.

## 📝 Lab note

Record, in your own words:

1. One number: the accuracy gap you measured between the training-distribution test set and the
   out-of-distribution test set in §3, and whether you consider that gap "small" or "concerning"
   for a real screening application.
2. One sentence: what distinguishes train/test leakage from ordinary domain shift — why are they
   different problems with different fixes?
3. One claim backed by a number: the accuracy gap between the "leaky" and "grouped" splits in §5,
   and what practical rule you would adopt in your own future work to avoid it.

## 📚 References

1. Simonyan, K., Vedaldi, A. & Zisserman, A. (2014). Deep Inside Convolutional Networks:
   Visualising Image Classification Models and Saliency Maps. *ICLR Workshop*.
2. Selvaraju, R. R. et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via
   Gradient-based Localization. *ICCV*.
3. Zech, J. R. et al. (2018). Variable generalization performance of a deep learning model to
   detect pneumonia in chest radiographs: a cross-sectional study. *PLOS Medicine*, 15(11).
4. Walsh, I. et al. (2021). DOME: recommendations for supervised machine learning validation in
   biology. *Nature Methods*, 18.
5. Varoquaux, G. & Cheplygina, V. (2022). Machine learning for medical imaging: methodological
   failures and recommendations for the future. *npj Digital Medicine*, 5.
6. Ribeiro, M. T., Singh, S. & Guestrin, C. (2016). "Why Should I Trust You?": Explaining the
   Predictions of Any Classifier. *KDD* (the LIME method, a model-agnostic alternative to the
   gradient- and occlusion-based attribution methods used in §2).
7. Kaufman, S., Rosset, S. & Perlich, C. (2012). Leakage in Data Mining: Formulation, Detection,
   and Avoidance. *KDD* (a general treatment of the leakage patterns demonstrated concretely in
   §5).

## ➡️ Next

Module 35 asks what happens once you start chaining many models and tools together — foundation
models (Module 28), profiling pipelines (Module 30), spatial analyses (Module 31) — under the
control of an AI agent, and why the auditing habits from this module become even more important
once a pipeline, not just a single model, is making decisions on your behalf.